# JupyterHub / Jupyter Notebook Unsloth Compatibility Check

This notebook helps you check whether your current JupyterHub or Jupyter Notebook environment is likely to support **Unsloth fine-tuning**.

It is designed for cases where you want to answer:

- Is my GPU capability high enough for Unsloth?
- Am I installing packages into the same Python environment as the current kernel?
- Is `unsloth` already installed?
- Can I import `FastLanguageModel` successfully?
- If something fails, what should I try next?

## What this notebook covers
1. Kernel / Python environment checks  
2. GPU and CUDA checks  
3. PyTorch checks  
4. Install-path / kernel mismatch checks  
5. Optional pip installation test  
6. Unsloth presence + import test  
7. A troubleshooting guide based on Unsloth docs and common GitHub issues

## Sources used for this notebook
This notebook reflects the current public installation guidance and common failure patterns from:

- Unsloth Zoo / installation notes
- Unsloth GitHub issues on Jupyter install mismatch, `unsloth_zoo` import errors, and kernel crashes

Please read the troubleshooting section near the end if your checks fail.

## 1. Check the current kernel and Python environment

A very common notebook problem is:

> You install packages into one Python environment, but your current Jupyter kernel is using a different Python environment.

That often causes:
- `ModuleNotFoundError`
- `pip install` appears to succeed, but `import unsloth` still fails

In [34]:
import os
import sys
import site
import platform
import subprocess

print("Python executable:", sys.executable)
print("Python version   :", sys.version)
print("Platform         :", platform.platform())
print("Current directory:", os.getcwd())

print("\nsite.getsitepackages():")
try:
    for p in site.getsitepackages():
        print(" -", p)
except Exception as e:
    print("Could not read site.getsitepackages():", repr(e))

print("\nsite.getusersitepackages():")
try:
    print(site.getusersitepackages())
except Exception as e:
    print("Could not read user site-packages:", repr(e))

Python executable: /opt/conda/bin/python3.12
Python version   : 3.12.8 | packaged by conda-forge | (main, Dec  5 2024, 14:24:40) [GCC 13.3.0]
Platform         : Linux-6.8.0-100-generic-x86_64-with-glibc2.39
Current directory: /home/jovyan/Small_Models_SP26

site.getsitepackages():
 - /opt/conda/lib/python3.12/site-packages

site.getusersitepackages():
/home/jovyan/.local/lib/python3.12/site-packages


In [35]:
print("which python:")
!which python

print("\npython --version:")
!python --version

print("\nwhich pip:")
!which pip

print("\npip --version:")
!pip --version

which python:
/opt/conda/bin/python

python --version:
Python 3.12.8

which pip:
/opt/conda/bin/pip

pip --version:
pip 25.0 from /opt/conda/lib/python3.12/site-packages/pip (python 3.12)


### How to interpret this section

You want the Python executable used by the notebook kernel and the Python used by `pip` to match as closely as possible.

A safer install command inside notebooks is:

```python
import sys
!{sys.executable} -m pip install <package>
```

That forces pip to install into the current kernel's Python environment.

In [36]:
import sys
print("Recommended install command for THIS kernel:")
print(f"{sys.executable} -m pip install <package>")

Recommended install command for THIS kernel:
/opt/conda/bin/python3.12 -m pip install <package>


## 2. Check whether the notebook sees an NVIDIA GPU

Unsloth's public installation notes say it supports **NVIDIA GPUs since 2018+** and lists a **minimum CUDA capability of 7.0** as the normal baseline. Linux and Windows are supported. If your notebook does not see an NVIDIA GPU at all, Unsloth fine-tuning is unlikely to work in this environment.

In [37]:
try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True,
        check=False
    )
    print("Return code:", result.returncode)

    if result.stdout.strip():
        print("\nstdout:")
        print(result.stdout)

    if result.stderr.strip():
        print("\nstderr:")
        print(result.stderr)
except FileNotFoundError:
    print("nvidia-smi command not found.")
except Exception as e:
    print("Error while running nvidia-smi:", repr(e))

Return code: 0

stdout:
Tue Apr 14 03:20:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1080 Ti     On  |   00000000:3D:00.0 Off |                  N/A |
|  0%   29C    P8             10W /  300W |       5MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------

## 3. Check whether PyTorch sees CUDA

This is one of the most important checks.

Even if `nvidia-smi` works, your notebook still needs a CUDA-enabled PyTorch build for fine-tuning.

In [38]:
try:
    import torch

    print("torch version            :", torch.__version__)
    print("torch.version.cuda       :", torch.version.cuda)
    print("cuda is available        :", torch.cuda.is_available())
    print("cuda device count        :", torch.cuda.device_count())

    if torch.cuda.is_available():
        print("device 0 name            :", torch.cuda.get_device_name(0))
        print("device 0 capability      :", torch.cuda.get_device_capability(0))
        print("device 0 total memory GB :", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
except Exception as e:
    print("PyTorch check failed:", repr(e))

torch version            : 2.5.1+cu124
torch.version.cuda       : 12.4
cuda is available        : True
cuda device count        : 1
device 0 name            : NVIDIA GeForce GTX 1080 Ti
device 0 capability      : (6, 1)
device 0 total memory GB : 10.9


## 4. Check whether your GPU capability meets the usual Unsloth baseline

Unsloth's public notes list **minimum CUDA capability 7.0** for the standard supported path.

Examples they list include:
- V100
- T4
- Titan V
- RTX 20 / 30 / 40 series
- A100 / H100 / L40

Older GPUs such as GTX 1070 / 1080 may work, but they are explicitly described as slower.

In [39]:
try:
    import torch

    if torch.cuda.is_available():
        capability = torch.cuda.get_device_capability(0)
        capability_value = capability[0] + capability[1] / 10
        print("Detected capability:", capability, "=>", capability_value)

        if capability_value >= 7.0:
            print("PASS: capability is >= 7.0")
        else:
            print("FAIL: capability is below 7.0")
    else:
        print("SKIP: CUDA is not available, so capability cannot be checked.")
except Exception as e:
    print("Capability check failed:", repr(e))

Detected capability: (6, 1) => 6.1
FAIL: capability is below 7.0


## 5. Optional: test whether package installation works in this kernel

This test uses the current kernel's Python interpreter directly.

It tries a small package install/upgrade (`ipykernel`) just to check whether:
- package installation is allowed
- pip is targeting the current kernel's environment
- the notebook environment is locked down by JupyterHub policy

In [40]:
import sys
cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "ipykernel"]
print("Running:", " ".join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True)
print("Return code:", result.returncode)

if result.stdout.strip():
    print("\nstdout:")
    print(result.stdout)

if result.stderr.strip():
    print("\nstderr:")
    print(result.stderr)

if result.returncode == 0:
    print("\nPASS: pip install appears to work in this kernel environment.")
else:
    print("\nFAIL: pip install did not succeed in this kernel environment.")

Running: /opt/conda/bin/python3.12 -m pip install -q --upgrade ipykernel
Return code: 0

PASS: pip install appears to work in this kernel environment.


In [41]:
import sys
!{sys.executable} -m pip install unsloth

## 6. Check whether `unsloth` is already installed

This only checks for the module's presence. It does **not** guarantee that imports will succeed, because some failures happen later during import when related dependencies are loaded.

In [42]:
import importlib.util

spec = importlib.util.find_spec("unsloth")
if spec is None:
    print("unsloth is NOT installed in the current kernel environment.")
else:
    print("unsloth appears to be installed.")
    print("Module origin:", spec.origin)

unsloth appears to be installed.
Module origin: /opt/conda/lib/python3.12/site-packages/unsloth/__init__.py


## 7. Try importing Unsloth

This is the real test.

Some environments report that `unsloth` is installed, but `from unsloth import FastLanguageModel` still fails because of:
- environment mismatch
- missing `unsloth_zoo`
- incompatible `vllm`
- Torch / CUDA / Triton / xformers conflicts

In [43]:
try:
    from unsloth import FastLanguageModel
    print("PASS: from unsloth import FastLanguageModel succeeded")
except Exception as e:
    print("FAIL: Unsloth import failed")
    print(repr(e))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
FAIL: Unsloth import failed
Exception("cannot import name '_is_tensor_or_array_like' from 'transformers.utils' (/opt/conda/lib/python3.12/site-packages/transformers/utils/__init__.py)")


/tmp/ipykernel_795/2039723976.py:2: UserWarning: WARNING: Unsloth should be imported before [trl, transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


## 8. If import failed, collect more environment information

Run this section before asking for help or before trying a reinstall.

In [44]:
packages_to_check = [
    "torch",
    "transformers",
    "trl",
    "peft",
    "accelerate",
    "bitsandbytes",
    "triton",
    "xformers",
    "vllm",
    "unsloth",
    "unsloth_zoo",
]

for pkg in packages_to_check:
    try:
        mod = __import__(pkg)
        version = getattr(mod, "__version__", "unknown")
        print(f"{pkg}: INSTALLED, version={version}")
    except Exception as e:
        print(f"{pkg}: NOT IMPORTABLE -> {repr(e)}")

torch: INSTALLED, version=2.5.1+cu124
transformers: INSTALLED, version=4.53.2
trl: INSTALLED, version=0.24.0
peft: NOT IMPORTABLE -> ValueError('Backend should be defined in the BACKENDS_MAPPING. Offending backend: mistral-common')
accelerate: INSTALLED, version=1.8.1
bitsandbytes: INSTALLED, version=0.49.2
triton: INSTALLED, version=3.6.0
xformers: NOT IMPORTABLE -> AttributeError("module 'torch' has no attribute 'float8_e8m0fnu'")
vllm: NOT IMPORTABLE -> ModuleNotFoundError("No module named 'vllm'")
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
unsloth: NOT IMPORTABLE -> Exception("cannot import name '_is_tensor_or_array_like' from 'transformers.utils' (/opt/conda/lib/python3.12/site-packages/transformers/utils/__init__.py)")
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
unsloth_zoo: NOT IMPORTABLE -> Exception("cannot import name '_is_tensor_or_array_like' from 'transformers.utils' (/opt/conda/lib/python3.12/site-packages/transfo

/tmp/ipykernel_795/610503546.py:17: UserWarning: WARNING: Unsloth should be imported before [trl, transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  mod = __import__(pkg)


In [45]:
print("pip list | grep -E 'unsloth|torch|transformers|trl|peft|accelerate|bitsandbytes|triton|xformers|vllm'")
!pip list | grep -E 'unsloth|torch|transformers|trl|peft|accelerate|bitsandbytes|triton|xformers|vllm'

pip list | grep -E 'unsloth|torch|transformers|trl|peft|accelerate|bitsandbytes|triton|xformers|vllm'
accelerate                         1.8.1
bitsandbytes                       0.49.2
fastrlock                          0.8.3
peft                               0.18.1
pytorch-ignite                     0.5.2
pytorch-lightning                  2.5.2
torch                              2.10.0
torch-geometric                    2.6.1
torchao                            0.17.0
torchaudio                         2.5.1+cu124
torchmetrics                       1.7.4
torchvision                        0.25.0
transformers                       5.5.0
triton                             3.6.0
trl                                0.24.0
unsloth                            2026.4.4
unsloth_zoo                        2026.4.6
xformers                           0.0.35


## 9. Suggested summary output

This cell prints a compact diagnosis.

In [46]:
summary = {}

# kernel python
summary["python_executable"] = sys.executable

# nvidia-smi
try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    summary["nvidia_smi_ok"] = (result.returncode == 0)
except Exception:
    summary["nvidia_smi_ok"] = False

# torch / cuda
try:
    import torch
    summary["torch_installed"] = True
    summary["torch_version"] = torch.__version__
    summary["torch_cuda_version"] = torch.version.cuda
    summary["cuda_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        cap = torch.cuda.get_device_capability(0)
        summary["gpu_name"] = torch.cuda.get_device_name(0)
        summary["gpu_capability"] = cap
        summary["gpu_capability_ok"] = (cap[0] + cap[1] / 10) >= 7.0
    else:
        summary["gpu_name"] = None
        summary["gpu_capability"] = None
        summary["gpu_capability_ok"] = False
except Exception as e:
    summary["torch_installed"] = False
    summary["torch_error"] = repr(e)

# unsloth installed
try:
    spec = importlib.util.find_spec("unsloth")
    summary["unsloth_installed"] = spec is not None
except Exception as e:
    summary["unsloth_installed"] = f"ERROR: {repr(e)}"

# unsloth import
try:
    from unsloth import FastLanguageModel
    summary["unsloth_import_ok"] = True
except Exception as e:
    summary["unsloth_import_ok"] = False
    summary["unsloth_import_error"] = repr(e)

for k, v in summary.items():
    print(f"{k}: {v}")

print("\nSuggested interpretation:")
if (
    summary.get("nvidia_smi_ok") is True
    and summary.get("cuda_available") is True
    and summary.get("gpu_capability_ok") is True
):
    print("Base hardware/runtime looks compatible with Unsloth.")
else:
    print("Base hardware/runtime does NOT yet look fully compatible with Unsloth.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
python_executable: /opt/conda/bin/python3.12
nvidia_smi_ok: True
torch_installed: True
torch_version: 2.5.1+cu124
torch_cuda_version: 12.4
cuda_available: True
gpu_name: NVIDIA GeForce GTX 1080 Ti
gpu_capability: (6, 1)
gpu_capability_ok: False
unsloth_installed: True
unsloth_import_ok: False
unsloth_import_error: Exception("cannot import name '_is_tensor_or_array_like' from 'transformers.utils' (/opt/conda/lib/python3.12/site-packages/transformers/utils/__init__.py)")

Suggested interpretation:
Base hardware/runtime does NOT yet look fully compatible with Unsloth.


/tmp/ipykernel_795/2081838168.py:42: UserWarning: WARNING: Unsloth should be imported before [trl, transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


# Troubleshooting guide

This section summarizes common failure patterns from Unsloth's public install notes and GitHub issues.

---

## Case A. `ModuleNotFoundError: No module named 'unsloth'`

### What it usually means
The package was installed into a different environment than the one your current kernel is using.

A public Unsloth GitHub issue shows this exact symptom after a notebook install. The issue text also includes a user report saying they fixed it by switching to terminal-based installation rather than installing from inside Jupyter.  
Source: Unsloth issue #1030.

### What to try
1. Check:
   - `sys.executable`
   - `which python`
   - `which pip`
2. Install with the current kernel explicitly:
   ```python
   import sys
   !{sys.executable} -m pip install unsloth
   ```
3. Safer approach:
   - create a fresh venv or conda env
   - install everything there
   - register it as a Jupyter kernel
   - switch the notebook to that kernel

### Recommended terminal workflow
```bash
python -m venv ~/venvs/unsloth_env
source ~/venvs/unsloth_env/bin/activate
python -m pip install --upgrade pip
python -m pip install ipykernel
python -m ipykernel install --user --name unsloth_env --display-name "Python (unsloth_env)"
```

Then select **Python (unsloth_env)** as your notebook kernel.

---

## Case B. `ImportError` involving `unsloth_zoo`

### What it usually means
Your `unsloth` and `unsloth_zoo` installation may be incomplete or inconsistent.

A public issue shows:
- `from unsloth import FastLanguageModel`
- then an import error mentioning `unsloth_zoo`
Source: Unsloth issue #1252.

### What to try
1. Reinstall both packages cleanly:
```bash
pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
```
2. Make sure the install is happening in the same Python environment as the current kernel.
3. Avoid mixing:
   - one notebook kernel
   - one terminal venv
   - another system-level pip

---

## Case C. Import fails because of `vllm` / `unsloth_zoo.vllm_utils`

### What it usually means
A public issue shows a failure path where importing Unsloth reaches:
- `unsloth_zoo.vllm_utils`
- then fails due to a compatibility problem involving `vllm`
Source: Unsloth issue #2977.

### What to try
1. Collect versions of:
   - `unsloth`
   - `unsloth_zoo`
   - `torch`
   - `transformers`
   - `trl`
   - `vllm`
2. Try a fresh isolated environment.
3. Prefer using the installation combinations documented by Unsloth for your CUDA + Torch version.
4. If you are doing basic SFT and do not need extra tooling, keep your environment as minimal as possible.

---

## Case D. The Jupyter kernel crashes or dies

### What it usually means
This often points to deeper binary / dependency issues rather than a simple Python import problem.

A public issue describes the notebook kernel dying and explicitly mentions trouble installing `xformers`.  
Source: Unsloth issue #613.

Another public issue shows a segmentation fault that happened in a local environment while the same workflow worked in Kaggle.  
Source: Unsloth issue #3185.

### What to try
1. Check whether `xformers` is installed and importable.
2. Check whether your Torch, CUDA, Triton, and xformers versions are mutually compatible.
3. Try a fresh environment instead of repairing a heavily modified existing one.
4. Compare against a known-good environment such as Kaggle / Colab if possible.
5. If this happens only in local Jupyter and not in Kaggle, do not assume your code is the problem. It may be an environment-level issue.

---

## Case E. Installation is weird or half-broken

Unsloth's public installation notes recommend the following escalation order:

1. First try a clean isolated environment:
```bash
python -m venv unsloth
source unsloth/bin/activate
pip install unsloth
```

2. If needed, install or verify:
- `torch`
- `triton`

3. Confirm CUDA is installed correctly:
```bash
nvcc --version
```

4. If needed, install `xformers` manually:
```bash
pip install ninja
pip install -v --no-build-isolation -U git+https://github.com/facebookresearch/xformers.git@main#egg=xformers
python -m xformers.info
```

5. For GRPO runs, test whether `vllm` installs successfully.

6. Double-check compatibility across:
- Python
- CUDA
- cuDNN
- torch
- triton
- xformers

7. Finally, verify bitsandbytes:
```bash
python -m bitsandbytes
```

---

## Case F. You want the exact recommended pip command for your Torch + CUDA combination

Unsloth's public installation page provides version-specific install commands, such as examples for:
- Torch 2.4 + CUDA 12.1
- Torch 2.5 + CUDA 12.4

It also provides an auto-install helper:

```bash
wget -qO- https://raw.githubusercontent.com/unslothai/unsloth/main/unsloth/_auto_install.py | python -
```

Use that only in a terminal you trust and only when your environment policy allows it.

---

# Practical recommendation

For JupyterHub and managed notebook systems, the safest setup is usually:

1. Get a GPU-backed notebook
2. Create a clean venv or conda env
3. Install Unsloth into that environment
4. Register that environment as a Jupyter kernel
5. Use that kernel consistently
6. Avoid mixing notebook installs, system installs, and multiple virtual environments

That setup will not solve every issue, but it avoids the most common causes of failure.

# Source notes

This notebook's guidance is based on the following public sources:

1. Unsloth Zoo GitHub page:
   - NVIDIA GPUs since 2018+
   - minimum CUDA capability 7.0
   - pip install / reinstall commands
   - advanced troubleshooting steps
   - version-specific pip install examples
   - auto-install helper

2. Unsloth GitHub issue #1030:
   - `ModuleNotFoundError: No module named 'unsloth'`
   - notebook install mismatch symptom
   - user-reported fix by installing from terminal

3. Unsloth GitHub issue #1252:
   - `unsloth_zoo` import-related failure

4. Unsloth GitHub issue #2977:
   - import path involving `unsloth_zoo.vllm_utils`
   - compatibility trouble involving `vllm`

5. Unsloth GitHub issue #613:
   - kernel crash / kernel dies in notebook environment
   - mentions xformers trouble

6. Unsloth GitHub issue #3185:
   - segmentation fault in local environment while the same workflow worked in Kaggle